In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


In [11]:
df = pd.read_csv("../../../../Merge/final_selected_data.csv")
df.drop(columns=["room_type_id", "hotel_id", "room_room_type_name", "hotel_name", "hotel_address"], inplace=True, errors='ignore')
df = df.dropna()
df = df.drop_duplicates()

TARGET_COL = "price_option_price"

numerical_cols = [
    'flexibility_score', 'sqm', 'bathroom_count', 
    'bedroom_count', 'adults_number', 'children_number'
]

binary_cols = [
    'large_double_bed', 'large_bed', 'single_bed', 'sofa_bed', 'double_bed',
    'small_double_bed', 'king_size_bed', 'futon_mattress', 'bunk_bed',
    'is_private_bathroom', 'balcony-terrace', 'closet', 'air_conditioning', 
    'hair_dryer', 'complimentary-bottled-water', 'bathtub', 'shower', 
    'refrigerator', 'high-floor', 'dressing-room', 'ground-floor', 'private-pool',
    'top-floor', 'complimentary-instant-coffee', 'electric-blanket', 
    'free-welcome-drink', 'low-floor', 'complimentary-tea', 'coffee-tea-maker', 
    'hot-spring-access', 'bãi_đậu_xe', 'phòng_tập', 'wifi_miễn_phí', 'không_hoàn_tiền', 
    'miễn_phí_hủy', 'vào_hồ_bơi_miễn_phí', 'đã_kèm_bữa_sáng'
]

categorical_cols = [
    "views",   # ví dụ: sea, mountain, city
    "region"
]

In [12]:
X = df[numerical_cols + binary_cols + categorical_cols]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)


In [13]:
def train_and_evaluate(model, model_name, use_pipeline=True):
    if use_pipeline:
        pipe = Pipeline(
            steps=[
                ("preprocess", preprocessor),
                ("model", model)
            ]
        )
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
    else:
        # dùng cho CatBoost (xử lý categorical trực tiếp)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


In [14]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

result_rf = train_and_evaluate(
    rf,
    "Random Forest"
)


In [15]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

result_xgb = train_and_evaluate(
    xgb,
    "XGBoost"
)


In [16]:
lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

result_lgbm = train_and_evaluate(
    lgbm,
    "LightGBM"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001494 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 371
[LightGBM] [Info] Number of data points in the train set: 14281, number of used features: 86
[LightGBM] [Info] Start training from score 1424629.989917


C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [17]:
cat_features = [X.columns.get_loc(col) for col in categorical_cols]

catboost = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    cat_features=cat_features
)

result_cat = train_and_evaluate(
    catboost,
    "CatBoost",
    use_pipeline=False
)


In [18]:
results = pd.DataFrame([
    result_rf,
    result_xgb,
    result_lgbm,
    result_cat
])

results


,model,MAE,RMSE,R2
0,Random Forest,563110.165195,3.723821e+06,0.223079
1,XGBoost,642036.915263,3.721470e+06,0.224060
2,LightGBM,800168.288158,3.853097e+06,0.168200
3,CatBoost,835481.122924,3.908848e+06,0.143955


In [19]:
df.shape

(17852, 46)